# Notebook 03 — Model Training and Evaluation
Train baseline classifiers on combined phishing features (manual + TF-IDF), compare results, save the best model, and export reusable prediction/report artifacts.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import sys, os
sys.path.append(os.path.abspath('..'))
from src.feature_extractor import extract_features

## Cell 2 — Load dataset and build feature matrix
Read `data/emails.csv`, transform each email text with `extract_features`, and assemble `X` and `y` for supervised training. The expected feature width is 3008.

In [3]:
df = pd.read_csv('../data/emails.csv')
X, y = [], []
for _, row in df.iterrows():
    vec, _ = extract_features(row['text'])
    X.append(vec)
    y.append(row['label'])
X = np.array(X)
y = np.array(y)
print(X.shape)  # must be (n_emails, 3008)

(3882, 3008)


## Cell 3 — Train/test split
Split the feature matrix into training and test sets. This creates an unseen holdout set for objective model comparison.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape) 
print(X_test.shape, y_test.shape)       

(3105, 3008) (3105,)
(777, 3008) (777,)


## Cell 4 — Train Logistic Regression baseline
Fit a linear baseline model and report train/test performance. This provides a strong, interpretable benchmark for text classification.

In [9]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_train_acc=accuracy_score(y_train, lr.predict(X_train))
lr_acc = accuracy_score(y_test, lr.predict(X_test))
lr_f1 = f1_score(y_test, lr.predict(X_test))
print(f"LR — Train Accuracy: {lr_train_acc:.3f}")
print(f"LR — Accuracy: {lr_acc:.3f}, F1: {lr_f1:.3f}")

LR — Train Accuracy: 0.987
LR — Accuracy: 0.967, F1: 0.953


## Cell 5 — Train Random Forest
Train a non-linear ensemble model and compute the same metrics for fair comparison against the baseline.

In [11]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_train_acc = accuracy_score(y_train, rf.predict(X_train))
rf_acc = accuracy_score(y_test, rf.predict(X_test))
rf_f1 = f1_score(y_test, rf.predict(X_test))
print(f"RF — Train Accuracy: {rf_train_acc:.3f}")   
print(f"RF — Accuracy: {rf_acc:.3f}, F1: {rf_f1:.3f}")

RF — Train Accuracy: 1.000
RF — Accuracy: 0.985, F1: 0.979


## Cell 6 — Compare models and inspect errors
Print side-by-side metrics, select the best model by F1 score, and inspect the confusion matrix to understand false positives/negatives.

In [12]:
print(f"{'Model':<22} {'Accuracy':>10} {'F1':>10}")
print("-" * 44)
print(f"{'Logistic Regression':<22} {lr_acc:>10.3f} {lr_f1:>10.3f}")
print(f"{'Random Forest':<22} {rf_acc:>10.3f} {rf_f1:>10.3f}")

best = lr if lr_f1 >= rf_f1 else rf
best_name = "Logistic Regression" if lr_f1 >= rf_f1 else "Random Forest"
print(f"\nBest model: {best_name}")
print("Confusion matrix:\n", confusion_matrix(y_test, best.predict(X_test)))

Model                    Accuracy         F1
--------------------------------------------
Logistic Regression         0.967      0.953
Random Forest               0.985      0.979

Best model: Random Forest
Confusion matrix:
 [[483   6]
 [  6 282]]


## Cell 7 — Save the best model
Persist the selected model to `outputs/model.pkl` so the frontend/API can reuse it for inference.

In [15]:
os.makedirs('../outputs', exist_ok=True)
joblib.dump(best, '../outputs/model.pkl')
print(f"Saved {best_name} with F1={max(lr_f1, rf_f1):.3f}")

Saved Random Forest with F1=0.979


## Cell 8 — Quick inference smoke test
Define a helper `predict()` function and run a couple of sample predictions to verify the saved model works end-to-end.

In [17]:
def predict(text):
    model = joblib.load('../outputs/model.pkl')
    vec, manual = extract_features(text)
    vec = vec.reshape(1, -1)
    label_int = model.predict(vec)[0]
    confidence = round(model.predict_proba(vec).max() * 100, 1)
    return {
        'label': 'Phishing' if label_int == 1 else 'Legitimate',
        'confidence': confidence,
        'features': manual
    }

print(predict("Urgent! Click here to verify your account now!!"))
print(predict("Hey, are we still on for lunch tomorrow?"))

{'label': 'Phishing', 'confidence': np.float64(96.0), 'features': {'url_count': 0, 'has_http_only': 0, 'suspicious_url_keywords': 0, 'urgent_keyword_count': 3, 'has_ip_url': 0, 'exclamation_count': 3, 'has_form_tag': 0, 'link_to_text_ratio': 0.0}}
{'label': 'Phishing', 'confidence': np.float64(79.0), 'features': {'url_count': 0, 'has_http_only': 0, 'suspicious_url_keywords': 0, 'urgent_keyword_count': 0, 'has_ip_url': 0, 'exclamation_count': 0, 'has_form_tag': 0, 'link_to_text_ratio': 0.0}}


## Cell 9 — Export reusable prediction module
Write `src/predict.py` so other components (e.g., backend) can import a stable `predict(text)` function.

In [19]:
%%writefile ../src/predict.py
import joblib, os
from src.feature_extractor import extract_features

def predict(text):
    model = joblib.load('../outputs/model.pkl')
    vec, manual = extract_features(text)
    vec = vec.reshape(1, -1)
    label_int = model.predict(vec)[0]
    confidence = round(model.predict_proba(vec).max() * 100, 1)
    return {
        'label': 'Phishing' if label_int == 1 else 'Legitimate',
        'confidence': confidence,
        'features': manual
    }

Overwriting ../src/predict.py


## Cell 10 — Export evaluation report
Generate a plain-text summary in `outputs/evaluation_report.txt` with final metrics and confusion matrix interpretation.

In [20]:
%%writefile ../outputs/evaluation_report.txt
Phishing Email Detector — Model Evaluation Report
==================================================

Model Comparison
----------------
Model                    Accuracy         F1
--------------------------------------------
Logistic Regression         0.967      0.953
Random Forest               0.985      0.979

Selected Model: Random Forest
Reason: Higher F1 score (0.979 vs 0.953), meaning fewer missed phishing emails.

Confusion Matrix (Random Forest)
---------------------------------
                  Predicted Legit    Predicted Phishing
Actual Legit               483                 6
Actual Phishing              6               282

Breakdown
---------
True Negatives  (TN): 483  — legitimate emails correctly identified
True Positives  (TP): 282  — phishing emails correctly caught
False Positives (FP):   6  — legitimate emails wrongly flagged as phishing
False Negatives (FN):   6  — phishing emails missed

Overall Performance
-------------------
Total test samples : 777
Correctly classified: 765
Misclassified       : 12
Final Accuracy      : 98.5%

Final F1 Score      : 0.979

Overwriting ../outputs/evaluation_report.txt
